# XRD value converter

In [1]:
# @title Read XRD data from "XRD input data" folder
!pip install -qU gspread

import pandas as pd
from pprint import pprint
import os

# Get from google drive folder
from google.colab import drive
drive.mount('/content/drive')

folder='/content/drive/MyDrive/Crossreads B D1/'
ifolder='XRD input data'
ipath = os.path.join(folder,ifolder)

# # @title Read CSV
df=pd.concat(
    pd.read_csv(os.path.join(ipath,ifn),sep=';')
    for ifn in os.listdir(ipath)
    if os.path.splitext(ifn)[-1].lower()=='.csv'
).fillna('')
paramcol = 'Parameter, Goal'

cols_to_ignore = {'Rwp','Rexp','Chi2','GOF'}
df=df[~df[paramcol].isin(cols_to_ignore)]


# clean sample
def clean_sample_num(x):
    if not x: return x
    x=x.strip().split()[0].split('-')[0]
    return ''.join(y for y in x if y.isdigit())

df['Sample']=df.Sample.apply(clean_sample_num)


# clean params
def clean_params(x):
    if x in {'Qcalcitemg', 'Qcalcitmg'}: return 'QMgCalcite'
    return x

df[paramcol]=df[paramcol].apply(clean_params)

df

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 786.3 kB/s eta 0:00:00
Mounted at /content/drive


,File,Sample,Sample ID,"Parameter, Goal",Value,ESD
0,C:/Users/Alessia/OneDrive - Nexus365/oxford/in...,000004,,Qcalcite,0.94720,0.0049
1,C:/Users/Alessia/OneDrive - Nexus365/oxford/in...,000004,,QMgCalcite,0.01670,0.0038
2,C:/Users/Alessia/OneDrive - Nexus365/oxford/in...,000004,,Qdolomite,0.03610,0.0032
7,C:/Users/Alessia/OneDrive - Nexus365/oxford/in...,000034,,Qcalcite,0.85400,0.01
8,C:/Users/Alessia/OneDrive - Nexus365/oxford/in...,000034,,QMgCalcite,0.11520,0.0097
...,...,...,...,...,...,...
211,C:/Users/Alessia/Dropbox/Taormina Mostra/XRD/x...,,taoSarcofago,Qalbint,0.00000,0.0
212,C:/Users/Alessia/Dropbox/Taormina Mostra/XRD/x...,,taoSarcofago,Qcalcite,0.98960,0.0016
213,C:/Users/Alessia/Dropbox/Taormina Mostra/XRD/x...,,taoSarcofago,Qdolomite,0.00450,0.001
214,C:/Users/Alessia/Dropbox/Taormina Mostra/XRD/x...,,taoSarcofago,Qmusc2m1,0.00350,0.0011


In [2]:
# @title Read "Crossreads B D1 new format" spreadsheet
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

spreadsheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1Yqxm6pwNAmz8GJDbcqOphNG9WB9xLif8u7sMplVIkqs/edit')

worksheet = spreadsheet.get_worksheet(0)
worksheet

# Get all data from the worksheet
rows = worksheet.get_all_values()
import pandas as pd
df_big = pd.DataFrame.from_records(rows)

# Set the first row as the header if needed
df_big.columns = df_big.iloc[0]
df_big = df_big.drop(0)

df_big = df_big.set_index(df_big.columns[0])
df_big['numeric_id']=[clean_sample_num(x) for x in df_big.index]
df_big=df_big.reset_index().set_index('numeric_id')
df_big

,reference or rock id,entity type,rock type,rock desc1,rock desc2,visually assessed granulometry (very fine < 1mm; fine < 2mm; medium 2-5mm; coarse > 5mm),visually assessed colour,visually assessed sharpness of veins,visually assessed colour of veins,visually assessed protolith features,...,microscopy / XRD: apatite,EPR Mn2+ ave,EPR Mn2+ stdevp,EPR dolomite %,EPR width ave,EPR width stdevp,EPR g=2.0044 ave,EPR g=2.0044 stdevp,EPR g=14.2515 ave,EPR g=14.2515 stdevp
numeric_id,,,,,,,,,,,,,,,,,,,,,
,Capedri et al 2004,reference,marble,carrara,,,,,,,...,,,,,,,,,,
,,,,,,,,,,,...,,,,,,,,,,
,Antonelli and Nestola 2021,reference,marble,carrara,,very fine,white/whitish,edgeless veins,grey to black,,...,,,,,,,,,,
,Antonelli and Nestola 2021,reference,marble,göktepe,,very fine,white/whitish,,,,...,,,,,,,,,,
,Antonelli and Lazzarini 2015,reference,marble,carrara,"fantiscritti, ravaccione, fossacava",very fine,white/whitish,edgeless and sharp veins,grey to black,no,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,EXMFT134,rock,marble,,,,,,,,...,,,,,,,,,,
005,ICT005,rock,marble,,,,,,,,...,,,,,,,,,,
006,ICT006,rock,marble,,,,,,,,...,,,,,,,,,,


In [11]:
# @title Update values in Crossreads sheet according to XRD data
cols = {
    'Qcalcite': 'XRD calcite content (%)',
    'QMgCalcite': 'XRD magnesian calcite content (%)',
    'Qdolomite': 'XRD dolomite content (%)',
    'SiO2p3221': 'XRD quartz content (%)',
    'quartz': 'XRD quartz content (%)',
    'musc2m1': 'XRD muscovite content (%)',
    'Aragonite': 'XRD aragonite content (%)',
    'Hematite': 'XRD Fe-oxihydroxides content (%)',
    'HEMATITE': 'XRD Fe-oxihydroxides content (%)',
    'Lepidocrocite': 'XRD Fe-oxihydroxides content (%)',
    'Goethite': 'XRD Fe-oxihydroxides content (%)',
    'PYRITE': 'XRD pyrite content (%)',
    'Kaolinite1A': 'XRD kaolinite content (%)',
    'Kaolinitedis': 'XRD kaolinite content (%)',
    'Kaolid': 'XRD kaolinite content (%)',
    'smectitedi2wfix1': 'XRD smectite content (%)',
    'Chlorite2b': 'XRD chlorite content (%)',
    'Glauconite': 'XRD glauconite content (%)',
    'Glauconite_1': 'XRD glauconite content (%)',
    'Glauconite_2': 'XRD glauconite content (%)',
    'Glauconite_3': 'XRD glauconite content (%)',
    'Orthoclase': 'XRD orthoclase content (%)',
    'Orthoclase_1': 'XRD orthoclase content (%)',
    'Orthoclase_2': 'XRD orthoclase content (%)',
    'Orthoclase_3': 'XRD orthoclase content (%)',
    'MicroInt1': 'XRD microcline content (%)',
    'MicroInt2': 'XRD microcline content (%)',
    'MicroMax': 'XRD microcline content (%)',
    'SANINA85': 'XRD Na-sanidine content (%)',
    'Sanina75': 'XRD Na-sanidine content (%)',
    'SANINA67': 'XRD Na-sanidine content (%)',
    'SANINA56': 'XRD Na-sanidine content (%)',
    'SANINA35': 'XRD Na-sanidine content (%)',
    'Sanina16': 'XRD Na-sanidine content (%)',
    'SANINA07': 'XRD Na-sanidine content (%)',
    'Sanid086': 'XRD K-sanidine content (%)',
    'Sanid08': 'XRD K-sanidine content (%)',
    'Anorthoclase_1': 'XRD anorthoclase content (%)',
    'Anorthoclase_2': 'XRD anorthoclase content (%)',
    'Anorthoclase_3': 'XRD anorthoclase content (%)',
    'Anorthoclase_4': 'XRD anorthoclase content (%)',
    'Albite': 'XRD albite content (%)',
    'ANORTK33': 'XRD albite content (%)',
    'ANORTK25': 'XRD albite content (%)',
    'ANORTK15': 'XRD albite content (%)',
    'MONALBIT': 'XRD albite content (%)',
    'ALBINT': 'XRD albite content (%)',
    'Analbite': 'XRD albite content (%)',
    'Oligoclase_1': 'XRD oligoclase content (%)',
    'Oligoclase_2': 'XRD oligoclase content (%)',
    'Oligoclase_3': 'XRD oligoclase content (%)',
    'Oligoclase_4': 'XRD oligoclase content (%)',
    'Plag16an': 'XRD oligoclase content (%)',
    'Plag25an': 'XRD oligoclase content (%)',
    'Andesine_1': 'XRD andesine content (%)',
    'Andesine_2': 'XRD andesine content (%)',
    'Plag50': 'XRD andesine content (%)',
    'PLAG50C1': 'XRD andesine content (%)',
    'Plag65an': 'XRD labradorite content (%)',
    'Labradorite_1': 'XRD labradorite content (%)',
    'Labradorite_2': 'XRD labradorite content (%)',
    'Labradorite_3': 'XRD labradorite content (%)',
    'Labradorite_4': 'XRD labradorite content (%)',
    'Labradorite_5': 'XRD labradorite content (%)',
    'Bytownite': 'XRD bytownite content (%)',
    'Plag85an': 'XRD bytownite content (%)',
    'Anorthite': 'XRD anorthite content (%)',
    'Anorthite_1': 'XRD anorthite content (%)',
    'Anorthite_2': 'XRD anorthite content (%)',
    'Anorthite_3': 'XRD anorthite content (%)',
    'Anorthite_4': 'XRD anorthite content (%)',
    'Anorthite_5': 'XRD anorthite content (%)',
    '*': 'XRD other minerals'
}

# Convert index to string and strip whitespace
df_big.index = df_big.index.astype(str).str.strip()

for i, row in df.iterrows():
    sample_id = str(row.Sample).strip()  # Convert to string and strip whitespace
    if sample_id not in set(df_big.index):
        print(f"!! {sample_id} not found in spreadsheet. Creating a new row.")
        # Create a new row with empty values
        new_row = pd.DataFrame({col: [''] for col in df_big.columns}, index=[sample_id])
        df_big = pd.concat([df_big, new_row])

    qcol = row[paramcol]
    if qcol in cols:
        vcol = cols[qcol]
        ecol = cols[qcol]+' ESD'
        df_big.at[sample_id, cols[qcol]] = row.Value * 100
        df_big.at[sample_id, cols[qcol]+' ESD'] = row.ESD * 100
    elif qcol not in str(df_big.at[sample_id, cols['*']]):
        current_value = str(df_big.at[sample_id, cols['*']])
        df_big.at[sample_id, cols['*']] = (current_value + ' ' + qcol).strip()

# Calculate combined columns
def sum_columns(row, columns):
    return sum(float(row.get(col, 0)) for col in columns if pd.notna(row.get(col)))

clay_minerals = ['XRD kaolinite content (%)', 'XRD smectite content (%)', 'XRD chlorite content (%)', 'XRD glauconite content (%)']
k_feldspar = ['XRD orthoclase content (%)', 'XRD microcline content (%)', 'XRD K-sanidine content (%)', 'XRD anorthoclase content (%)']
plagioclase = ['XRD albite content (%)', 'XRD oligoclase content (%)', 'XRD andesine content (%)', 'XRD labradorite content (%)', 'XRD bytownite content (%)', 'XRD anorthite content (%)']

df_big['XRD clay minerals'] = df_big.apply(lambda row: sum_columns(row, clay_minerals), axis=1)
df_big['XRD K-feldspar'] = df_big.apply(lambda row: sum_columns(row, k_feldspar), axis=1)
df_big['XRD plagioclase'] = df_big.apply(lambda row: sum_columns(row, plagioclase), axis=1)

id='001399'
print(f'Values updated. Here for example are new values for ISic{id}:\n')
d=dict(df_big.loc[id])
columns_to_show = list(cols.values()) + ['XRD clay minerals', 'XRD K-feldspar', 'XRD plagioclase']
pprint({k:v for k,v in d.items() if k in set(columns_to_show)})

# Sort the DataFrame by index before updating the worksheet
df_big = df_big.sort_index()
df_big
# res=worksheet.update([df_big.columns.values.tolist()] + df_big.values.tolist())
# if not isinstance(res, dict) or not (res.get('spreadsheetId') and res.get('updatedCells')):
#     print('!! Error updating Google Sheets worksheet')
# else:
#     print(f"Successfully updated {res['updatedCells']} cells in the Google Sheets worksheet.")

Values updated. Here for example are new values for ISic001399:

{'XRD K-feldspar': 0,
 'XRD calcite content (%)': '',
 'XRD clay minerals': 0,
 'XRD dolomite content (%)': 32.7,
 'XRD magnesian calcite content (%)': 34.8,
 'XRD other minerals': ' Qmusc2m1 Qhematite QIron QOligoclase Qquartz '
                       'QMirabilite QPhlogopite1M',
 'XRD plagioclase': 0}


,reference or rock id,entity type,rock type,rock desc1,rock desc2,visually assessed granulometry (very fine < 1mm; fine < 2mm; medium 2-5mm; coarse > 5mm),visually assessed colour,visually assessed sharpness of veins,visually assessed colour of veins,visually assessed protolith features,...,EPR width ave,EPR width stdevp,EPR g=2.0044 ave,EPR g=2.0044 stdevp,EPR g=14.2515 ave,EPR g=14.2515 stdevp,XRD clay minerals,XRD K-feldspar,XRD plagioclase,0
,Capedri et al 2004,reference,marble,carrara,,,,,,,...,,,,,,,0,0,0,NaN
,Lazzarini and Mariottini 1987,reference,marble,thasian,limin,,,,,,...,,,,,,,0,0,0,NaN
,Lazzarini and Mariottini 1987,reference,marble,thasian,skira,,,,,,...,,,,,,,0,0,0,NaN
,Lazzarini and Mariottini 1987,reference,marble,aphrodisian,,,,,,,...,,,,,,,0,0,0,NaN
,Lazzarini and Mariottini 1987,reference,marble,proconnesian,marmara,,,,,,...,,,,,,,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
visually assessed colour of veins,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
visually assessed granulometry (very fine < 1mm; fine < 2mm; medium 2-5mm; coarse > 5mm),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
visually assessed monomineralic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
visually assessed protolith features,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN
